# CS261 Mini Project 2 — Semantic Segmentation on Pascal VOC 2007

**Setup flow:**
1. Clone the GitHub repo
2. Install dependencies
3. Download the dataset (via Kaggle API)
4. Verify the dataset loads correctly

## 1. Clone the repository

Replace the URL with your own GitHub repo URL.

In [ ]:
import os

REPO_URL = "https://github.com/gracee-chen/261-mini2.git"
REPO_DIR = "261-mini2"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Download Pascal VOC 2007 dataset via Kaggle API

**Prerequisites:** upload your `kaggle.json` API token.  
Get it from https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
# Upload kaggle.json
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

In [ ]:
# Place the token and download the dataset
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download (~1.9 GB)
!kaggle datasets download -d zaraks/pascal-voc-2007 --unzip -p .

# Verify the directory exists
!ls -lh VOCtrainval_06-Nov-2007/VOCdevkit/VOC2007/

### Alternative: mount from Google Drive

If you already have the dataset on Drive, use this instead of the Kaggle block above.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# VOC_ROOT = "/content/drive/MyDrive/VOCtrainval_06-Nov-2007"  # adjust path
# os.environ["VOC_ROOT"] = VOC_ROOT

## 4. Set dataset root path

In [ ]:
import os, sys

VOC_ROOT = os.path.join(os.getcwd(), "VOCtrainval_06-Nov-2007")
os.environ["VOC_ROOT"] = VOC_ROOT

# Make dataset/ importable
sys.path.insert(0, os.path.join(os.getcwd(), "dataset"))

print("VOC_ROOT:", VOC_ROOT)

## 5. Verify dataset loading

In [ ]:
import torch
from voc_dataset import VOC_CLASSES, NUM_CLASSES, get_datasets, get_dataloaders, mask_to_class_index

print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {VOC_CLASSES}")

train_ds, val_ds = get_datasets(VOC_ROOT)
print(f"\nTrain samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}  (used as test set)")

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Inspect a batch
train_loader, val_loader = get_dataloaders(VOC_ROOT, batch_size=4)

images, masks = next(iter(train_loader))
print(f"Image batch shape : {images.shape}")   # (4, 3, 256, 256)
print(f"Mask  batch shape : {masks.shape}")    # (4, 1, 256, 256)

## 6. Visualise sample images and masks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_sample(image, mask, title=""):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img_np = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    mask_np = mask_to_class_index(mask).numpy()
    mask_display = mask_np.copy()
    mask_display[mask_display == 255] = 0

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    if title:
        fig.suptitle(title, fontsize=12)

    axes[0].imshow(img_np)
    axes[0].set_title("Input Image")
    axes[0].axis("off")

    seg_map = axes[1].imshow(mask_display, cmap="tab20", vmin=0, vmax=20)
    axes[1].set_title("Segmentation Mask (cleaned)")
    axes[1].axis("off")

    cbar = plt.colorbar(seg_map, ax=axes[1], ticks=range(21))
    cbar.ax.set_yticklabels([f"{i}: {VOC_CLASSES[i]}" for i in range(21)])
    cbar.ax.tick_params(labelsize=7)

    plt.tight_layout()
    plt.show()

# Show 2 random training samples
import random
for idx in random.sample(range(len(train_ds)), 2):
    img, mask = train_ds[idx]
    classes_present = [VOC_CLASSES[c] for c in np.unique(mask_to_class_index(mask).numpy()) if c != 255]
    print(f"Sample {idx} — classes: {classes_present}")
    show_sample(img, mask, title=f"Train sample {idx}")